In this notebook, we build three models that predict the match result and frames win rate of professional snooker matches:  
1. using elo rating system.
2. Linear model with all features
3. Linear model with 3 selected features.


In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('/Users/tliu/Desktop/Erdos Project/3_Player_Data_Generation/match_data_50_tourns_modified.csv')

In [3]:
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_played_3_years,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage
0,Cao Yupeng,Jiang Jun,9,1411,1044,0.918016,0.714532,341,179,2302,...,446,242,32,17,32,17,5,0,0.0,1.000000
1,Siripaporn Nuanthakhamjan,Zhou Yuelong,9,965,1384,0.056881,0.259705,3,0,23,...,23,4,245,118,808,430,2,5,1.0,0.285714
2,Wu Yize,Allan Taylor,9,1342,1237,0.656987,0.565251,78,37,552,...,404,215,101,49,336,147,5,3,0.0,0.625000
3,Ben Woollaston,Oliver Brown,9,1412,1146,0.845318,0.660383,803,454,5053,...,534,282,91,35,235,97,5,2,0.0,0.714286
4,Andres Petrov,Mark Williams,9,1045,1611,0.017765,0.195447,51,18,304,...,167,61,281,170,1112,649,2,5,1.0,0.285714


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5058 entries, 0 to 5057
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player1                   5058 non-null   object 
 1   player2                   5058 non-null   object 
 2   best_of                   5058 non-null   int64  
 3   player1_elo               5058 non-null   int64  
 4   player2_elo               5058 non-null   int64  
 5   elo_match_win_rate        5058 non-null   float64
 6   elo_frame_win_rate        5058 non-null   float64
 7   p1_matches_played         5058 non-null   int64  
 8   p1_matches_won            5058 non-null   int64  
 9   p1_frames_played          5058 non-null   int64  
 10  p1_frames_won             5058 non-null   int64  
 11  p2_matches_played         5058 non-null   int64  
 12  p2_matches_won            5058 non-null   int64  
 13  p2_frames_played          5058 non-null   int64  
 14  p2_frame

In [5]:
#Add more features
data['p1_frames_win_rate'] = data['p1_frames_won']/ data['p1_frames_played']
data['p2_frames_win_rate'] = data['p2_frames_won']/ data['p2_frames_played']

data['p1_matches_win_rate'] = data['p1_matches_won']/ data['p1_matches_played']
data['p2_matches_win_rate'] = data['p2_matches_won']/ data['p2_matches_played']

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5058 entries, 0 to 5057
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player1                   5058 non-null   object 
 1   player2                   5058 non-null   object 
 2   best_of                   5058 non-null   int64  
 3   player1_elo               5058 non-null   int64  
 4   player2_elo               5058 non-null   int64  
 5   elo_match_win_rate        5058 non-null   float64
 6   elo_frame_win_rate        5058 non-null   float64
 7   p1_matches_played         5058 non-null   int64  
 8   p1_matches_won            5058 non-null   int64  
 9   p1_frames_played          5058 non-null   int64  
 10  p1_frames_won             5058 non-null   int64  
 11  p2_matches_played         5058 non-null   int64  
 12  p2_matches_won            5058 non-null   int64  
 13  p2_frames_played          5058 non-null   int64  
 14  p2_frame

In [7]:
#p1_matches_win_rate and p2_matches_win_rate both have missing values
#We will fill them with 0.5
data.fillna(0.5, inplace = True)

In [11]:
data.describe()

,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,p1_frames_won,p2_matches_played,...,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage,p1_frames_win_rate,p2_frames_win_rate,p1_matches_win_rate,p2_matches_win_rate
count,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,...,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000,5058.000000
mean,8.165480,1273.810004,1275.520364,0.497275,0.498917,505.810004,300.055951,3476.342626,1893.389877,508.892448,...,505.286279,269.042507,3.292210,3.289245,0.500000,0.502254,0.492700,0.492426,0.495471,0.494568
std,3.914968,196.492484,195.600071,0.232470,0.116075,510.943055,338.405206,3731.600388,2142.316593,513.352541,...,373.824231,217.974509,2.278517,2.303192,0.500049,0.295306,0.086260,0.085231,0.146481,0.146111
min,1.000000,636.000000,617.000000,0.001432,0.133542,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,7.000000,1104.000000,1108.000000,0.322129,0.421285,57.000000,24.000000,328.000000,149.250000,62.000000,...,180.000000,81.000000,2.000000,1.000000,0.000000,0.285714,0.471962,0.470588,0.444444,0.444444
50%,7.000000,1301.000000,1303.000000,0.498633,0.499063,321.000000,155.000000,1993.000000,985.500000,327.000000,...,464.000000,231.000000,3.000000,3.000000,0.500000,0.500000,0.508348,0.507022,0.516736,0.513514
75%,9.000000,1425.750000,1425.000000,0.668204,0.575054,890.750000,499.000000,6052.500000,3252.000000,892.250000,...,772.000000,414.000000,4.000000,4.000000,1.000000,0.714286,0.541006,0.540425,0.583333,0.583454
max,35.000000,1695.000000,1695.000000,0.995452,0.855388,1930.000000,1361.000000,14957.000000,8816.000000,1930.000000,...,1898.000000,1166.000000,18.000000,18.000000,1.000000,1.000000,0.800000,0.800000,1.000000,1.000000


In [8]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2,
                                        shuffle = False)

In [9]:
#Create two dictionaries to record the scores.
win_perc_pred_scores = {}

match_result_pred_scores = {}

***

## Model1: Using elo rating to predict winner and frame win percentage

In [10]:
#Model1: Using elo rating to predict winner and frame win percentage
from sklearn.metrics import accuracy_score, root_mean_squared_error

pred_train = data_train['elo_match_win_rate']<0.5
score1_train = accuracy_score(pred_train, data_train['match_result'])

pred_test = data_test['elo_match_win_rate']<0.5
score1_test = accuracy_score(pred_test, data_test['match_result'])

score2_train = root_mean_squared_error(data_train['elo_frame_win_rate'], data_train['win_percentage'])

score2_test = root_mean_squared_error(data_test['elo_frame_win_rate'], data_test['win_percentage'])

#Record the scores
win_perc_pred_scores['elo'] = score2_test
match_result_pred_scores['elo'] = score1_test

print('The prediction accuracy of the match results on the training set is:', score1_train)
print('The rmse on win percentage on the training set:', score2_train)
print('The prediction accuracy of the match results on the test set is:', score1_test)
print('The rmse on win percentage on the test set:', score2_test)

The prediction accuracy of the match results on the training set is: 0.6873455264458724
The rmse on win percentage on the training set: 0.26713032300820494
The prediction accuracy of the match results on the test set is: 0.6462450592885376
The rmse on win percentage on the test set: 0.27173425159629216


### Is there a relation between the prediction accuracy and number the matches played by the players?

In [11]:
df = data_test.copy()
df['pred_result'] = pred_test == data_test['match_result']
corr_df = df[df['pred_result']]
inc_df = df.drop(corr_df.index, axis = 0)
corr_matches_played = pd.concat([corr_df['p1_matches_played'], corr_df['p2_matches_played']], axis = 0, ignore_index=True)
inc_matches_played = pd.concat([inc_df['p1_matches_played'], inc_df['p2_matches_played']], axis = 0, ignore_index=True)

In [12]:
corr_matches_played.describe()

count    1308.000000
mean      424.301988
std       513.318426
min         0.000000
25%        32.000000
50%       206.000000
75%       677.500000
max      1930.000000
dtype: float64

In [13]:
inc_matches_played.describe()

count     716.000000
mean      443.655028
std       523.998414
min         0.000000
25%        18.000000
50%       206.000000
75%       827.250000
max      1930.000000
dtype: float64

The correct prediction and incorrect prediction have similar distribution of number ofmatches played. It seems that there is no obvious relation between the prediction accuracy and number the matches played by the players.

## Model2: Linear model with all features

In [14]:
#Create predictors and targets for training and test set
result_train = data_train['match_result']
win_perc_train = data_train['win_percentage']
X_train = data_train.drop(['match_result', 'win_percentage', 'player1', 'player2', 'score1', 'score2'], axis = 1)


result_test = data_test['match_result']
win_perc_test = data_test['win_percentage']
X_test = data_test.drop(['match_result', 'win_percentage', 'player1', 'player2', 'score1', 'score2'], axis = 1)

In [15]:
#Linear Regression with all features
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
pipe_linear = Pipeline([('scale', StandardScaler()),
                    ('lr', LinearRegression())])

pipe_linear.fit(X_train, win_perc_train)
linear_pred = pipe_linear.predict(X_test)

#calculate rmse
lr_score1 = root_mean_squared_error(linear_pred, win_perc_test)

pipe_logit = Pipeline([('scale', StandardScaler()),
                    ('lr', LogisticRegression())])
pipe_logit.fit(X_train, result_train)
logit_pred = pipe_logit.predict(X_test)

#Calculate the accuracy score
lr_score2 = accuracy_score(logit_pred, result_test)

#Record the scores
win_perc_pred_scores['linear'] = lr_score1
match_result_pred_scores['linear'] = lr_score2

print('The prediction accuracy of the match results on the test set is:', lr_score2)
print('The rmse on win percentage on the test set is:', lr_score1)


The prediction accuracy of the match results on the test set is: 0.6403162055335968
The rmse on win percentage on the test set is: 0.26681672481770397


/Users/tliu/miniconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Linear model has very similar performance as predicting by elo rating.

## Model3: Linear model with selected features

In [16]:
#Linear model using selected features picked by intuitions
selected_features1 = ['elo_frame_win_rate', 'p1_frames_win_rate', 'p2_frames_win_rate']

X_train1 = X_train[selected_features1]
X_test1 = X_test[selected_features1]
pipe_linear = Pipeline([('scale', StandardScaler()),
                    ('lr', LinearRegression())])

pipe_linear.fit(X_train1, win_perc_train)
linear_pred = pipe_linear.predict(X_test1)
lr2_score1 = root_mean_squared_error(linear_pred, win_perc_test)


selected_features2 = ['elo_match_win_rate', 'p1_matches_win_rate', 'p2_matches_win_rate']
X_train2 = X_train[selected_features2]
X_test2 = X_test[selected_features2]
pipe_logit = Pipeline([('scale', StandardScaler()),
                    ('lr', LogisticRegression())])
pipe_logit.fit(X_train2, result_train)
logit_pred = pipe_logit.predict(X_test2)
lr2_score2 = accuracy_score(logit_pred, result_test)

#Record the scores
win_perc_pred_scores['linear with selected features'] = lr2_score1
match_result_pred_scores['linear with selected features'] = lr2_score2

print('The prediction accuracy of the match results on the test set is:', lr2_score2)
print('The rmse on win percentage on the test set is:', lr2_score1)

The prediction accuracy of the match results on the test set is: 0.642292490118577
The rmse on win percentage on the test set is: 0.26662362433899306


## Model4: PCA + Linear

In [17]:
from sklearn.decomposition import PCA

n_components = list(range(1,26))

n_1 = 0
n_2 = 0

score1_best = 1
score2_best = 0

for n in n_components: 
    pipe_pca = Pipeline([('scale', StandardScaler()),
                            ('pca', PCA(n)),
                        ('lr', LinearRegression())])

    pipe_pca.fit(X_train, win_perc_train)
    linear_pred = pipe_pca.predict(X_test)
    score1 = root_mean_squared_error(linear_pred, win_perc_test)
    if score1 < score1_best:
        score1_best = score1
        n_1 = n

    pipe_pca2 = Pipeline([('scale', StandardScaler()),
                           ('pca', PCA(n)),
                        ('lr', LogisticRegression(max_iter = 10000))])
    pipe_pca2.fit(X_train, result_train)
    logit_pred = pipe_pca2.predict(X_test)
    score2 = accuracy_score(logit_pred, result_test)

    if score2 > score2_best:
        score2_best = score2
        n_2 = n

#Record the best scores
win_perc_pred_scores[f'pca({n_1})'] = score1_best
match_result_pred_scores[f'pca({n_2})'] = score2_best


In [18]:
print('Performance (rmse) on predicting frames win rate percentage:\n', win_perc_pred_scores)
print('Performance on predicting match result:\n', match_result_pred_scores)

Performance (rmse) on predicting frames win rate percentage:
 {'elo': 0.27173425159629216, 'linear': 0.26681672481770397, 'linear with selected features': 0.26662362433899306, 'pca(22)': 0.2663953839002387}
Performance on predicting match result:
 {'elo': 0.6462450592885376, 'linear': 0.6403162055335968, 'linear with selected features': 0.642292490118577, 'pca(7)': 0.642292490118577}


They have similar performance in predicting frames win percentage. However, the linear model with 3 selected features do poorly in predicting the winner comparing to the other two.